# C - Additional metrics & loss curves (Clusters 7 & 11)

**Reviewer comments addressed**

* **R1.6 / R2.6 (Cluster 7)** - only relative L2 is reported; add RMSE, MAE,
  maximum absolute error (and normalized error).
* **R2.4 (Cluster 11)** - no loss-convergence curves (PDE / BC / total).

Bonus (reviewer R3 #6, pointwise-error map): we also show the **absolute**
error map and a **global-max-normalized** relative error map, which do not blow
up near near-zero field values - addressing the "uninformative pointwise error"
comment.

In [ ]:
# === Environment setup (robust: local / Colab native / VSCode-Colab) ===
# Run this cell FIRST. It is idempotent (safe to re-run) and does NOT touch
# Colab's preinstalled numpy/scipy/torch (avoids ABI breakage) - it just puts
# the repo's `src/` on the path and installs the one missing dep, scikit-fem.
import os, sys, subprocess, importlib
REPO_URL = 'https://github.com/Daniel14gonc/PINNs_piezoelectricity.git'

def _find_repo():
    """Return the repo root (folder with src/pinn_piezo) at/above cwd."""
    d = os.getcwd()
    for _ in range(10):
        if os.path.isdir(os.path.join(d, 'src', 'pinn_piezo')):
            return d
        nd = os.path.dirname(d)
        if nd == d:
            return None
        d = nd
    return None

repo = _find_repo()
if repo is None:
    # Fresh runtime: clone ONCE into a fixed absolute path (no nesting on re-run).
    base = '/content' if os.path.isdir('/content') else os.getcwd()
    repo = os.path.join(base, 'PINNs_piezoelectricity')
    if not os.path.isdir(os.path.join(repo, 'src', 'pinn_piezo')):
        subprocess.run(['git', 'clone', REPO_URL, repo], check=True)
    else:
        subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=False)

os.chdir(repo)
src = os.path.join(repo, 'src')
if src not in sys.path:
    sys.path.insert(0, src)   # import from source; NO pip install of the stack

# Only scikit-fem is missing on a stock Colab; install it (its deps already exist).
try:
    import skfem  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'scikit-fem'], check=True)

# Verify the revision modules made it into the checkout (i.e. you pushed them).
missing = [m for m in ('pinn_piezo', 'pinn_piezo.fem', 'pinn_piezo.metrics',
                       'pinn_piezo.indirect.standard')
           if importlib.util.find_spec(m) is None]
assert not missing, ('Missing modules: ' + ', '.join(missing) +
                     ' - push them to GitHub, then `git -C ' + repo +
                     ' pull` (or restart runtime) and re-run.')

import torch, pinn_piezo
print('pinn_piezo :', pinn_piezo.__file__)
print('repo       :', repo)
print('torch      :', torch.__version__, '| cuda:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Where this notebook writes its tables / figures.
from pathlib import Path
OUT = Path('outputs/revision'); OUT.mkdir(parents=True, exist_ok=True)
print('Artefacts ->', OUT.resolve())

In [ ]:
# The geometry/collocation datasets (data/*.npy) are gitignored, so a fresh
# clone does not have them. Regenerate the paper-size sets if missing
# (n_collocation=150 -> 150**2 = 22,500 collocation points).
import os
from pinn_piezo import geometry
for suffix in ('_m1', '_m1_d'):
    if not os.path.exists(f'data/xy_top_non_normalized{suffix}.npy'):
        geometry.generate_and_save(n_points=400, n_collocation=150,
                                   n_collocation_test=200, suffix=suffix,
                                   data_dir='data')
        print('generated data', suffix)
    else:
        print('data present', suffix)

## 1. Loss curves: PDE loss, BC loss, total (Cluster 11)

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
torch.set_default_dtype(torch.float64)
from pinn_piezo import geometry
from pinn_piezo.indirect import model as imodel, train as itrain
from pinn_piezo.indirect.losses import physics_loss, get_BC_loss

QUICK = True
import os
EP_ADAM  = int(os.environ.get('REV_ADAM',  300 if QUICK else 1000))
EP_LBFGS = int(os.environ.get('REV_LBFGS', 50  if QUICK else 200))

arrays = itrain.load_dataset('data', suffix='_m1', fraction=1.0)
tensors = itrain.to_device(arrays, DEVICE, dtype=torch.float64)

# Instrumented training loop that records each loss component per epoch.
np.random.seed(0); torch.manual_seed(0)
model = imodel.build_default_model(device=DEVICE, model_type='pyramid')

def components():
    bc = get_BC_loss(tensors['xy_top'], tensors['xy_bottom'],
                     tensors['xy_right'], tensors['xy_left'], model)
    pde = physics_loss(tensors['x_collocation'], tensors['y_collocation'],
                       model, tensors['coefficients'])
    return pde, bc

hist = {'pde': [], 'bc': [], 'total': []}
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for ep in range(EP_ADAM):
    opt.zero_grad(); pde, bc = components(); loss = pde + bc
    loss.backward(); opt.step()
    hist['pde'].append(pde.item()); hist['bc'].append(bc.item()); hist['total'].append(loss.item())

opt = torch.optim.LBFGS(model.parameters(), lr=1e-2)
for ep in range(EP_LBFGS):
    def closure():
        opt.zero_grad(); pde, bc = components(); l = pde + bc; l.backward(); return l
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
    opt.step(closure)
    pde, bc = components()
    hist['pde'].append(pde.item()); hist['bc'].append(bc.item()); hist['total'].append((pde+bc).item())

np.savez(OUT / 'loss_curves_indirect.npz', **hist)

In [ ]:
plt.figure(figsize=(7,4.5))
for k, c in [('pde','tab:blue'), ('bc','tab:orange'), ('total','k')]:
    plt.semilogy(hist[k], label=f'{k} loss', color=c, lw=1.5)
plt.axvline(EP_ADAM, ls='--', color='gray', alpha=.7, label='Adam -> L-BFGS')
plt.xlabel('Epoch'); plt.ylabel('Loss (log scale)')
plt.title('Indirect PINN - loss convergence'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / 'loss_curves_indirect.png', dpi=200); plt.show()

## 2. Full metric table: RMSE / MAE / max / L2 (Cluster 7)

In [ ]:
from pinn_piezo import metrics, fem
from pinn_piezo.config import MODELS_DIR
from pinn_piezo.indirect.train import tensorize

# Reference (indirect).
# Reference fields from the validated scikit-fem solver (Notebook 00 shows it
# matches the analytical solution; poling_sign=-1 = this repo's indirect convention).
r = fem.solve_piezo('indirect', nx=300, ny=10, voltage=100.0, poling_sign=-1.0)
df = pd.DataFrame({'X_Coordinate': r.points[:,0], 'Y_Coordinate': r.points[:,1],
                   'X_Deflection': r.u, 'Y_Deflection': r.v, 'Potential': r.phi})
XY = df[['X_Coordinate','Y_Coordinate']].values
REF = {'u': df['X_Deflection'].values, 'v': df['Y_Deflection'].values, 'phi': df['Potential'].values}

# Use the paper's pretrained model for the reported numbers.
paper = imodel.build_default_model(device=DEVICE, model_type='pyramid')
paper.load_state_dict(torch.load(MODELS_DIR / 'indirect' / 'model_PINN_indirect_paper_3.pt',
                                 map_location=DEVICE))
paper.eval()
p = paper(tensorize(XY, DEVICE)).detach().cpu().numpy()
table = metrics.metrics_table({'u':p[:,0],'v':p[:,1],'phi':p[:,2]}, REF)
table.to_csv(OUT / 'metrics_indirect.csv')
print('Indirect (note: ~0.19-0.20 rel.L2 for u,v means ~20% error - qualify '
      '"high accuracy" language accordingly!)')
table

## 3. Better pointwise-error maps (bonus, reviewer R3 #6)

In [ ]:
u_err_abs = np.abs(p[:,0] - REF['u'])
# global-max normalized relative error (bounded, unlike local-denominator).
u_err_rel = u_err_abs / (np.max(np.abs(REF['u'])) + 1e-30)

fig, ax = plt.subplots(1, 2, figsize=(12, 2.6))
for a, val, title in [(ax[0], u_err_abs, 'Absolute error |u_pred - u|'),
                      (ax[1], u_err_rel, 'Error / max|u|  (global-normalized)')]:
    sc = a.scatter(XY[:,0], XY[:,1], c=val, cmap='jet', s=8); a.set_title(title)
    fig.colorbar(sc, ax=a)
plt.tight_layout(); plt.savefig(OUT / 'error_maps_u.png', dpi=200); plt.show()

---
### Rebuttal snippet (Clusters 7 & 11)
> *We added convergence curves for the PDE, boundary and total losses (Fig …)
> and report RMSE, MAE and maximum absolute error in addition to the relative
> L2 (Table …). We also replaced the local-denominator pointwise error with the
> absolute error and a global-max-normalized relative error, which remain
> bounded near near-zero field values. We have qualified the accuracy language:
> the electric potential is highly accurate (~0.1%), while the displacement
> fields carry ~20% relative L2 error, now discussed explicitly.*